# Expanding Training Set Reduced
The following Jupyter notebook is meant to merge the analysis for the various expanding training set functions from the affilitated notebooks, specifically regarding the reduced dataset. Repeat each of the following steps for the three main classifiers: Decision Tree, Logistic Regression, and Neural Net at every prediction week
1. Tuned the hyperparameters to ensure the most efficient model is utilized
2. Saving the specific performance metrics (Sensitivity, Specificity, Accuracy, Normalized MCC) 
3. Calculating the SHAP files for the various 9 columns
4. Saving each model to a .pkl file for every week

The various performace metrics and the SHAP values are saved to pickle, per classifier.


## Load Dependencies

In [1]:
#%reset
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn import tree
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

from sklearn.metrics import accuracy_score, confusion_matrix, matthews_corrcoef
from num2words import num2words
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, KFold, RepeatedStratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score, average_precision_score
import word2number
from word2number import w2n
from sklearn.tree import DecisionTreeClassifier
import pickle
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import RocCurveDisplay
import random
from matplotlib.patches import Polygon
import shap
import os

from Functions import prep_training_test_data_period, prep_training_test_data, calculate_metrics,cross_validation_leave_geo_out, prep_training_test_data_shifted, add_labels_to_subplots, LOOCV_by_HSA_dataset, save_in_HSA_dictionary, prepare_data_and_model
hfont = {'fontname':'Helvetica'}
palette = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#e5c494']
import json 

## Load Data

In [2]:
HSA_weekly_data_all = pd.read_csv("HSA_Weekly_Surveillance.csv")

columns_to_remove = [col for col in HSA_weekly_data_all.columns if 'cases' in col]
HSA_weekly_data_no_cases_no_deaths = HSA_weekly_data_all.drop(columns=columns_to_remove)

columns_to_remove = [col for col in HSA_weekly_data_no_cases_no_deaths.columns if 'deaths' in col]
HSA_weekly_data_no_cases_no_deaths = HSA_weekly_data_no_cases_no_deaths.drop(columns=columns_to_remove)

HSA_weekly_data_all.loc[1, 'week_fifty-two_beds_over_15_100k']

0.0

## Define Global Variables

In [3]:
no_iterations = 100
geography_column = 'HSA_ID'
geo_split = 0.9
time_period = 'period'  # Choose 'period', 'exact', or 'shifted'
size_of_test_dataset = 1
train_weeks_for_initial_model = 1
weeks_to_predict = range(1, 123 - size_of_test_dataset - 3 - train_weeks_for_initial_model)

weeks_in_future = 3
weight_col = 'weight'
keep_output = True

# This is for SHAP Analysis - the order of the column names in the X_test files
feature_names=['COVID-19 admissions', 
               'COVID-19 ICU beds', 
               'COVID-19 hospital beds', 
               'Perc. beds with \nCOVID-19 patients', 
               '\u0394 COVID-19 admissions',
               '\u0394 COVID-19 ICU beds',
               '\u0394 COVID-19 hospital beds',
               '\u0394 Perc. beds with \nCOVID-19 patients',
               '> 15 per 100,000 COVID-19 \npatients in hospital beds']

# Just to check
print(len(feature_names))

9


## Location and County Data

In [4]:
## County Data 
data_by_county = pd.read_csv('county_time_data_all_dates.csv')

data_by_county = data_by_county.dropna(subset=['admits_weekly', 'deaths_weekly', 'cases_weekly', 'icu_weekly', 'beds_weekly', 'perc_covid'])
data_by_county['CTYNAME'] = data_by_county['CTYNAME'].apply(lambda x: x.split()[0])
data_by_county['CTYNAME'] = data_by_county['fips'].astype(str) + '' + data_by_county['CTYNAME']
data_by_county['beds_over_15_100k'] = (data_by_county['beds_weekly'] > 15) * 1

# Redo dates
for i, week in enumerate(data_by_county['date'].unique()):
    data_by_county.loc[data_by_county['date'] == week, 'week'] = i

## DELTA POLYGON 
start_date = pd.to_datetime('2021-06-30')
end_date = pd.to_datetime('2021-10-26')
data_by_county['date'] = pd.to_datetime(data_by_county['date'])
for i, week in enumerate(data_by_county['date'].unique()):
    data_by_county.loc[data_by_county['date'] == week, 'week'] = i
# Find the indices of rows that match the exact start and end dates
matching_indices_start = data_by_county.loc[data_by_county['date'] <= start_date].index.max()
matching_indices_end = data_by_county.loc[data_by_county['date'] <= end_date].index.max()
first_week_delta = data_by_county.loc[matching_indices_start, 'week']
last_week_delta = data_by_county.loc[matching_indices_end, 'week']
start_date = pd.to_datetime('2021-10-26')
end_date = pd.to_datetime('2022-09-27')
data_by_county['date'] = pd.to_datetime(data_by_county['date'])
for i, week in enumerate(data_by_county['date'].unique()):
    data_by_county.loc[data_by_county['date'] == week, 'week'] = i
# Find the indices of rows that match the exact start and end dates
matching_indices_start = data_by_county.loc[data_by_county['date'] <= start_date].index.max()
matching_indices_end = data_by_county.loc[data_by_county['date'] <= end_date].index.max()
first_week_omricon = data_by_county.loc[matching_indices_start, 'week']
last_week_omricon = data_by_county.loc[matching_indices_end, 'week']

## CDC POLYGON 
start_date = pd.to_datetime('2021-03-01')
end_date = pd.to_datetime('2022-01-24')
data_by_county['date'] = pd.to_datetime(data_by_county['date'])
for i, week in enumerate(data_by_county['date'].unique()):
    data_by_county.loc[data_by_county['date'] == week, 'week'] = i
# Find the indices of rows that match the exact start and end dates
matching_indices_start = data_by_county.loc[data_by_county['date'] <= start_date].index.max()
matching_indices_end = data_by_county.loc[data_by_county['date'] <= end_date].index.max()
first_week_CDC = data_by_county.loc[matching_indices_start, 'week']
last_week_CDC = data_by_county.loc[matching_indices_end, 'week']

/var/folders/b1/ts1cmy7n6kg0gzvxmtp8cc_80000gn/T/ipykernel_23221/873074510.py:2: DtypeWarning: Columns (47,48,49,50,51,55,56) have mixed types. Specify dtype option on import or set low_memory=False.
  data_by_county = pd.read_csv('county_time_data_all_dates.csv')


## Adding Percent Exceeding Capacity

In [5]:
percent_exceed_capacity = []

# Iterate through the columns of the DataFrame
for column_name in HSA_weekly_data_all.columns:
    if 'beds_over_15_100k' in column_name:
        # Calculate the sum of the column and append it to the list
        column_sum = HSA_weekly_data_all[column_name].sum() / len(HSA_weekly_data_all[column_name])
        percent_exceed_capacity.append(column_sum)

## Define Hyperparameter Tuning Functions

In [6]:
def hyperparameter_train_model_DT(X_train, y_train, sample_weights, decision_tree):   
    # Here are the hyperparamters
    param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': np.arange(2, 5, 1)}
    
    # Now we construct the grid of hyperparamters to train
    grid_search = GridSearchCV(estimator=decision_tree,
                           param_grid=param_grid,
                           scoring='average_precision',  # Optimize average_precision
                           cv=5,
                           n_jobs=-1, 
                           verbose=1) 
    # Now evaluate
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)
    
    # Now we return the best model for usage
    best_model = grid_search.best_estimator_
    
    return best_model


def hyperparameter_train_model_LR(X_train, y_train, sample_weights, logistic):   
    # Here are the hyperparameters
    param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100], 
    'penalty': ['l1', 'l2'],           
    'solver': ['liblinear', 'saga'],    
    'class_weight': [None, 'balanced']  
}
    # Now we construct the grid of hyperparamters to train
    grid_search = GridSearchCV(estimator=logistic,
                           param_grid=param_grid,
                           scoring='average_precision',  
                           cv=5,
                           n_jobs=-1, 
                           verbose=1) 
    # Now evaluate
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)
    
    # Now we return the best model for usage
    best_model = grid_search.best_estimator_
    
    return best_model

def hyperparameter_train_model_NN(X_train, y_train, sample_weights, mlp):
    param_grid = {
        'hidden_layer_sizes': [(64,), (128,), (256,)],
        'learning_rate_init': [0.001, 0.01, 0.0001],
        'learning_rate': ['adaptive', 'constant'],
    }

    grid_search = GridSearchCV(
        estimator=mlp,
        param_grid=param_grid,
        scoring='average_precision',   # same as your LR helper; swap to 'f1_weighted' if preferred
        cv=5,
        n_jobs=-1,      # use all cores like LR helper
        verbose=1
    )

    grid_search.fit(X_train, y_train)
    
    return grid_search.best_estimator_

def hyperparameter_train_model_RF(X_train, y_train, sample_weights):   
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [2, 4, 6],
        'min_samples_split': [2, 5, 10]
    }

    # Construct the grid search
    grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                               param_grid=param_grid,
                               scoring='average_precision',
                               cv=5,
                               n_jobs=-1,
                               verbose=1)

    # Evaluate and fit the model
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)

    # Return the best model
    return grid_search.best_estimator_


## Preparing Training Function

In [7]:
def prep_training_test_data(
    data, no_weeks, weeks_in_future, geography, weight_col, keep_output
):
    ## Get the weeks for the x and y datasets
    x_weeks = []
    y_weeks = []
    for week in no_weeks:
        test_week = int(week) + weeks_in_future
        x_weeks.append("_" + num2words(week) + "_")
        y_weeks.append("_" + num2words(test_week) + "_")
    X_data = pd.DataFrame()
    y_data = pd.DataFrame()
    weights_all = pd.DataFrame()
    missing_data = []
    ## Now get the training data
    k = 0
    for x_week in x_weeks:
        y_week = y_weeks[k]
        k += 1
        weeks_x = [col for col in data.columns if x_week in col]
        columns_x = [geography] + weeks_x + [weight_col]
        data_x = data[columns_x]
        weeks_y = [col for col in data.columns if y_week in col]
        columns_y = [geography] + weeks_y
        data_y = data[columns_y]
        # ensure they have the same amount of data
        # remove rows in test_data1 with NA in test_data2
        data_x = data_x.dropna()
        data_x = data_x[data_x[geography].isin(data_y[geography])]
        # remove rows in test_data2 with NA in test_data1
        data_y = data_y.dropna()
        data_y = data_y[data_y[geography].isin(data_x[geography])]
        data_x = data_x[data_x[geography].isin(data_y[geography])]
        data_x_no_HSA = len(data_x[geography].unique())
        missing_data.append(
            (
                (len(data[geography].unique()) - data_x_no_HSA)
                / len(data[geography].unique())
            )
            * 100
        )
        # get weights
        # weights = weight_data[weight_data[geography].isin(data_x[geography])][[geography, weight_col]]
        X_week = data_x.iloc[:, 1 : len(columns_x)]  # take away y, leave weights for mo
        y_week = data_y.iloc[:, -1]
        y_week = y_week.astype(int)
        weights = X_week.iloc[:, -1]
        if keep_output:
            X_week = X_week.iloc[
                :, : len(X_week.columns) - 1
            ]  # remove the weights and leave "target" for that week
            # rename columns for concatenation
            X_week.columns = range(1, len(data_x.columns) - 1)
        else:
            X_week = X_week.iloc[
                :, : len(X_week.columns) - 2
            ]  # remove the weights and  "target" for that week
            X_week.columns = range(
                1, len(data_x.columns) - 2
            )  # remove the weights and  "target" for that week
            # rename columns for concatenation
        y_week.rename("0", inplace=True)
        X_data = pd.concat([X_data, X_week], axis = 0)
        y_data = pd.concat([y_data, y_week], axis = 0)
        weights.rename("0", inplace=True)
        weights_all = pd.concat([weights_all, weights], axis = 0)
    X_data.reset_index(drop=True, inplace=True)
    y_data.reset_index(drop=True, inplace=True)
    weights_all.reset_index(drop=True, inplace=True)
    return (X_data, y_data, weights_all, missing_data)

## Preparing Folder for the Models

In [8]:
# Preparing folders for the different models
dt_folder = "decisionTreeFolder"
lr_folder = "logisticRegressionFolder"
nn_folder = "neuralNetworkFolder"

# Double check in-case folders are not prepared
def create_directory_if_not_exists(directory_path):
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        print(f"Created directory: {directory_path}")
    else:
        print(f"Directory already exists: {directory_path}")

# Now apply to the folder names
create_directory_if_not_exists(dt_folder)
create_directory_if_not_exists(lr_folder)
create_directory_if_not_exists(nn_folder)

Directory already exists: decisionTreeFolder
Directory already exists: logisticRegressionFolder
Directory already exists: neuralNetworkFolder


## Prepare Data (Reduced)

In [9]:
reduced_regex_pattern = (
    "HSA|weight|beds_over_15_100k|"
    "admits|icu|beds|perc_covid"
)

columns_to_select_reduced = HSA_weekly_data_all.filter(
    regex=reduced_regex_pattern
).columns.tolist()

# Making sure we do not have the cases
columns_to_select_reduced = [col for col in columns_to_select_reduced if "cases" not in col and "deaths" not in col]


reduced_data = HSA_weekly_data_all[columns_to_select_reduced]

In [10]:
# reduced_data.columns.tolist()

['HSA_ID',
 'week_one_admits',
 'week_one_icu',
 'week_one_beds',
 'week_one_perc_covid',
 'week_one_admits_delta',
 'week_one_icu_delta',
 'week_one_beds_delta',
 'week_one_perc_covid_delta',
 'week_one_beds_over_15_100k',
 'week_two_admits',
 'week_two_icu',
 'week_two_beds',
 'week_two_perc_covid',
 'week_two_admits_delta',
 'week_two_icu_delta',
 'week_two_beds_delta',
 'week_two_perc_covid_delta',
 'week_two_beds_over_15_100k',
 'week_three_admits',
 'week_three_icu',
 'week_three_beds',
 'week_three_perc_covid',
 'week_three_admits_delta',
 'week_three_icu_delta',
 'week_three_beds_delta',
 'week_three_perc_covid_delta',
 'week_three_beds_over_15_100k',
 'week_four_admits',
 'week_four_icu',
 'week_four_beds',
 'week_four_perc_covid',
 'week_four_admits_delta',
 'week_four_icu_delta',
 'week_four_beds_delta',
 'week_four_perc_covid_delta',
 'week_four_beds_over_15_100k',
 'week_five_admits',
 'week_five_icu',
 'week_five_beds',
 'week_five_perc_covid',
 'week_five_admits_delta',


## Loop - Logisitic Regression

In [15]:
weeks_to_predict = range(1, 123 - size_of_test_dataset - 3 - train_weeks_for_initial_model)
ROC_by_week_lr_period = []
PRC_by_week_lr_period = [] # This is for the PRC graph
sensitivity_by_week_lr_period = []
specificity_by_week_lr_period = []
ppv_by_week_lr_period = []
npv_by_week_lr_period = []
accuracy_by_week_lr_period = []
norm_MCC_by_week_lr_period = []
neo_sensitivity_by_week_lr_period = []
neo_specificity_by_week_lr_period = []
size_of_test_dataset = 1
for prediction_week in weeks_to_predict:
    print(prediction_week)
    print(range(1 , int(prediction_week + train_weeks_for_initial_model) + 1))
    print(range(int(prediction_week + train_weeks_for_initial_model) + 1, int(prediction_week + train_weeks_for_initial_model + size_of_test_dataset) + 1))

    X_train_lr, y_train_lr, weights_lr, missing_data_train_HSA = prep_training_test_data(reduced_data, no_weeks=range(1, int(prediction_week + train_weeks_for_initial_model) + 1), weeks_in_future=3, geography='HSA_ID', weight_col='weight', keep_output=True)
    

    X_test_lr, y_test_lr, weights_test_lr, missing_data_test_HSA = prep_training_test_data(reduced_data, no_weeks=range(int(prediction_week + train_weeks_for_initial_model) + 1, int(prediction_week + train_weeks_for_initial_model + size_of_test_dataset) + 1), weeks_in_future=3, geography='HSA_ID', weight_col='weight', keep_output=True)
    
    
    weights_lr = weights_lr.to_numpy()
    weights_lr = weights_lr.ravel()
    
    # Instantiate classifier
    clf_lr =  DecisionTreeClassifier(random_state=10) 
    # Hyperparamter train
    best_clf_lr = hyperparameter_train_model_DT(X_train_lr, y_train_lr, weights_lr, clf_lr)
    
    # Now we save in our folder
    # Create filename
    # model_to_save_lr = os.path.join(lr_folder, f"Reduced_model_week_{prediction_week}.pkl")
    
    # Saving the model using pickle
    # with open(model_to_save_lr, "wb") as f:
        # pickle.dump(best_clf_lr, f)

    # Make predictions on the test set
    y_pred = best_clf_lr.predict(X_test_lr)
    y_pred_proba = best_clf_lr.predict_proba(X_test_lr)

    # Evaluate the accuracy of the model
    accuracy_by_week_lr_period.append(accuracy_score(y_test_lr, y_pred))
    if len(np.unique(y_test_lr)) > 1:
    # Calculate ROC AUC score only if there are multiple classes
        ROC_by_week_lr_period.append(roc_auc_score(y_test_lr, y_pred_proba[:, 1]))
        PRC_by_week_lr_period.append(average_precision_score(y_test_lr, y_pred_proba[:, 1]))
    else:
        ROC_by_week_lr_period.append(np.nan)
        PRC_by_week_lr_period.append(np.nan)
    try:
        conf_matrix = confusion_matrix(y_test_lr, y_pred)
        FP = conf_matrix[0, 1]
        FN = conf_matrix[1, 0]
        sensitivity, specificity, ppv, npv = calculate_metrics(conf_matrix)
        sensitivity_by_week_lr_period.append(sensitivity)
        specificity_by_week_lr_period.append(specificity)
    
        ppv_by_week_lr_period.append(ppv)
        npv_by_week_lr_period.append(npv)
        
        neo_sensitivity_by_week_lr_period.append(sensitivity + specificity - 1)
        neo_specificity_by_week_lr_period.append(FP + FN)
    except:
        sensitivity_by_week_lr_period.append(np.nan)
        specificity_by_week_lr_period.append(np.nan)
    
        ppv_by_week_lr_period.append(np.nan)
        npv_by_week_lr_period.append(np.nan)
        
        neo_sensitivity_by_week_lr_period.append(np.nan)
        neo_specificity_by_week_lr_period.append(np.nan)
    
# Uncomment for SHAP - takes ~20/30 minutes
'''
    # Time for SHAP Analysis LR
    X_test_lr.columns = feature_names
    
    # Need to pass background data for distribution estimation
    explainer_lr = shap.LinearExplainer(best_clf_lr, X_train_lr)
    shap_values_lr = explainer_lr(X_test_lr)
# Concatenate all values
    if prediction_week == weeks_to_predict[0]:
        shap_values_all_lr = shap_values_lr
    else: # Concatenate the SHAP values
        for feature in X_test_lr.columns: 
                shap_values_all_lr.values = np.concatenate([shap_values_all_lr.values, shap_values_lr.values])
                shap_values_all_lr.base_values = np.concatenate([shap_values_all_lr.base_values, shap_values_lr.base_values])
                shap_values_all_lr.data = np.concatenate([shap_values_all_lr.data, shap_values_lr.data])
'''


1
range(1, 3)
range(3, 4)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
2
range(1, 4)
range(4, 5)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
3
range(1, 5)
range(5, 6)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
4
range(1, 6)
range(6, 7)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
5
range(1, 7)
range(7, 8)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
6
range(1, 8)
range(8, 9)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
7
range(1, 9)
range(9, 10)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
8
range(1, 10)
range(10, 11)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
9
range(1, 11)
range(11, 12)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
10
range(1, 12)
range(12, 13)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
11
range(1, 13)
range(13, 14)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
12
range(1, 14)
range(14, 15)
Fitting 5

'\n    # Time for SHAP Analysis LR\n    X_test_lr.columns = feature_names\n    \n    # Need to pass background data for distribution estimation\n    explainer_lr = shap.LinearExplainer(best_clf_lr, X_train_lr)\n    shap_values_lr = explainer_lr(X_test_lr)\n# Concatenate all values\n    if prediction_week == weeks_to_predict[0]:\n        shap_values_all_lr = shap_values_lr\n    else: # Concatenate the SHAP values\n        for feature in X_test_lr.columns: \n                shap_values_all_lr.values = np.concatenate([shap_values_all_lr.values, shap_values_lr.values])\n                shap_values_all_lr.base_values = np.concatenate([shap_values_all_lr.base_values, shap_values_lr.base_values])\n                shap_values_all_lr.data = np.concatenate([shap_values_all_lr.data, shap_values_lr.data])\n'

## Saving Logistic Regression Performance Metrics

In [16]:
# Organize the file names
lr_performance_name = "reduced_lr_performance_stats.pkl"
# lr_SHAP_name = "lr_SHAP.pkl"

# Organize into a dictionary
lr_performance_stats = {
    "ROC": ROC_by_week_lr_period,
    "PRC": PRC_by_week_lr_period,
    "accuracy": accuracy_by_week_lr_period,
    "sensitivity": sensitivity_by_week_lr_period,
    "specificity": specificity_by_week_lr_period,
    "ppv": ppv_by_week_lr_period,
    "npv": npv_by_week_lr_period,
    "MCC": norm_MCC_by_week_lr_period,
    "neo_sensitivity": neo_sensitivity_by_week_lr_period,
    "neo_specificity": neo_specificity_by_week_lr_period
}

# Open
with open(lr_performance_name, 'wb') as f:
    pickle.dump(lr_performance_stats, f)
    
'''
with open(lr_SHAP_name, 'wb') as f:
    pickle.dump(shap_values_all_lr, f)
'''

"\nwith open(lr_SHAP_name, 'wb') as f:\n    pickle.dump(shap_values_all_lr, f)\n"

## Loop - Neural Network

In [ ]:
weeks_to_predict = range(1, 123 - size_of_test_dataset - 3 - train_weeks_for_initial_model)
ROC_by_week_nn_period = []
PRC_by_week_nn_period = [] # This is for the PRC graph
sensitivity_by_week_nn_period = []
specificity_by_week_nn_period = []
ppv_by_week_nn_period = []
npv_by_week_nn_period = []
accuracy_by_week_nn_period = []
norm_MCC_by_week_nn_period = []

size_of_test_dataset = 1
for prediction_week in weeks_to_predict:
    print(prediction_week)
    print(range(1 , int(prediction_week + train_weeks_for_initial_model) + 1))
    print(range(int(prediction_week + train_weeks_for_initial_model) + 1, int(prediction_week + train_weeks_for_initial_model + size_of_test_dataset) + 1))

    X_train_nn, y_train_nn, weights_nn, missing_data_train_HSA = prep_training_test_data(reduced_data, no_weeks=range(1, int(prediction_week + train_weeks_for_initial_model) + 1), weeks_in_future=3, geography='HSA_ID', weight_col='weight', keep_output=True)
    

    X_test_nn, y_test_nn, weights_test_nn, missing_data_test_HSA = prep_training_test_data(reduced_data, no_weeks=range(int(prediction_week + train_weeks_for_initial_model) + 1, int(prediction_week + train_weeks_for_initial_model + size_of_test_dataset) + 1), weeks_in_future=3, geography='HSA_ID', weight_col='weight', keep_output=True)
    
    weights_nn = weights_nn.to_numpy()
    weights_nn = weights_nn.ravel()
    
    # Instantiate classifier
    clf_nn = MLPClassifier(max_iter=1000)

    # Hyperparamter train - no sample weights as handled by SMOTE
    best_clf_nn = hyperparameter_train_model_NN(X_train_nn, y_train_nn, weights_nn, clf_nn)
    
    model_to_save_nn = os.path.join(nn_folder, f"Reduced_model_week_{prediction_week}.pkl")
    
    # Saving the model using pickle
    with open(model_to_save_nn, "wb") as f:
        pickle.dump(best_clf_nn, f)

    # Make predictions on the test set
    y_pred = best_clf_nn.predict(X_test_nn)
    y_pred_proba = best_clf_nn.predict_proba(X_test_nn)

    # Evaluate the accuracy of the model
    accuracy_by_week_nn_period.append(accuracy_score(y_test_nn, y_pred))
    if len(np.unique(y_test_nn)) > 1:
    # Calculate ROC AUC score only if there are multiple classes
        ROC_by_week_nn_period.append(roc_auc_score(y_test_nn, y_pred_proba[:, 1]))
        PRC_by_week_nn_period.append(average_precision_score(y_test_nn, y_pred_proba[:, 1]))
    else:
        ROC_by_week_nn_period.append(np.nan)
        PRC_by_week_nn_period.append(np.nan)
    conf_matrix = confusion_matrix(y_test_nn, y_pred)

    sensitivity, specificity, ppv, npv = calculate_metrics(conf_matrix)
    sensitivity_by_week_nn_period.append(sensitivity)
    specificity_by_week_nn_period.append(specificity)

    ppv_by_week_nn_period.append(ppv)
    npv_by_week_nn_period.append(npv)

    norm_MCC_by_week_nn_period.append((matthews_corrcoef(y_test_nn, y_pred) + 1)/2)

# Uncomment for SHAP - takes ~3/4 hours
'''
    # Time for SHAP Analysis NN
    X_test_nn.columns = feature_names
    X_train_nn.columns = feature_names
    
    # Need to pass background data for distribution estimation
    explainer_nn = shap.KernelExplainer(best_clf_nn.predict_proba, X_train_nn)
    shap_values_nn = explainer_nn(X_test_nn)

    # Concatenate all values
    if prediction_week == weeks_to_predict[0]:
        shap_values_all_nn = shap_values_nn
    else: # Concatenate the SHAP values
        for feature in X_test_nn.columns: 
                shap_values_all_nn.values = np.concatenate([shap_values_all_nn.values, shap_values_nn.values])
                shap_values_all_nn.base_values = np.concatenate([shap_values_all_nn.base_values, shap_values_nn.base_values])
                shap_values_all_nn.data = np.concatenate([shap_values_all_nn.data, shap_values_nn.data])
'''


## Saving Neural Net Performance Metrics

In [14]:
nn_performance_name = "reduced_nn_performance_stats.pkl"
nn_SHAP_name = "nn_SHAP.pkl"

# Organize into a dictionary
nn_performance_stats = {
    "ROC": ROC_by_week_nn_period,
    "PRC": PRC_by_week_nn_period,
    "accuracy": accuracy_by_week_nn_period,
    "sensitivity": sensitivity_by_week_nn_period,
    "specificity": specificity_by_week_nn_period,
    "ppv": ppv_by_week_nn_period,
    "npv": npv_by_week_nn_period,
    "MCC": norm_MCC_by_week_nn_period
}

# Open
with open(nn_performance_name, 'wb') as f:
    pickle.dump(nn_performance_stats, f)

# with open(nn_SHAP_name, 'wb') as f:
    # pickle.dump(shap_values_all_nn, f)